## -------------------------------------------------------------------------
## NOME DO PROGRAMA: Sistema de Inspeção de Qualidade SPV - RedVase
## DATA DO PROGRAMA: 10/04/2026
## EQUIPE: Vision-ABC
## Grupo: 8
## INTEGRANTES:
##   - Ronaldo Ávila de Arruda Junior, RA: 11202231908
##   - Leonardo Garcia dos Santos, RA: 11202230441
##
## EXEMPLO DE CHAMADA:
##   python3 main.py
## -------------------------------------------------------------------------

In [15]:
import cv2
import numpy as np
import os
import datetime

# --- Variáveis Globais de Calibração ---
sensibilidade = 0
tolerancia_px = 40
area_min_quebra = 50

# Cria o diretório para salvar as falhas
if not os.path.exists('falhas'):
    os.makedirs('falhas')

def controle_mouse(event, x, y, flags, param):
    """
    Detecta cliques do mouse na tela para ajustar os valores.
    Coordenadas (x, y) precisam bater com os retângulos desenhados na interface.
    """
    global sensibilidade, tolerancia_px, area_min_quebra
    
    if event == cv2.EVENT_LBUTTONDOWN:
        # Linha 1: Sensibilidade (Y entre 20 e 50)
        if 20 <= y <= 50:
            if 300 <= x <= 340: sensibilidade = max(0, sensibilidade - 1)
            elif 360 <= x <= 400: sensibilidade = min(20, sensibilidade + 1)
            
        # Linha 2: Tolerância Superfície (Y entre 60 e 90)
        elif 60 <= y <= 90:
            if 300 <= x <= 340: tolerancia_px = max(0, tolerancia_px - 5)
            elif 360 <= x <= 400: tolerancia_px = min(300, tolerancia_px + 5)
            
        # Linha 3: Área Mínima de Quebra (Y entre 100 e 130)
        elif 100 <= y <= 130:
            if 300 <= x <= 340: area_min_quebra = max(0, area_min_quebra - 5)
            elif 360 <= x <= 400: area_min_quebra = min(500, area_min_quebra + 5)

def desenhar_painel_sobreposto(frame):
    """
    Desenha o painel de controle escuro com botões sobre a imagem da câmera.
    """
    # Fundo do painel (Cinza escuro) e Borda
    cv2.rectangle(frame, (10, 10), (420, 145), (30, 30, 30), -1)
    cv2.rectangle(frame, (10, 10), (420, 145), (200, 200, 200), 1) 

    # Configuração dos textos e posições Y
    linhas = [
        (f"Sensib. BlackHat: {sensibilidade}", 40, 20),
        (f"Tol. Superficie (px): {tolerancia_px}", 80, 60),
        (f"Area Min Quebra (px): {area_min_quebra}", 120, 100)
    ]

    for texto, y_texto, y_botao in linhas:
        # Escreve o nome do parâmetro
        cv2.putText(frame, texto, (20, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # Botão Vermelho [ - ]
        cv2.rectangle(frame, (300, y_botao), (340, y_botao + 30), (0, 0, 180), -1)
        cv2.putText(frame, "-", (312, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # Botão Verde [ + ]
        cv2.rectangle(frame, (360, y_botao), (400, y_botao + 30), (0, 180, 0), -1)
        cv2.putText(frame, "+", (372, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
    return frame

# --- FUNÇÕES DE PROCESSAMENTO MANTIDAS ---
def segmentar_vermelho(img_bgr):
    img_suavizada = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv = cv2.cvtColor(img_suavizada, cv2.COLOR_BGR2HSV)
    lower1, upper1 = np.array([0, 30, 20]), np.array([10, 255, 255])
    lower2, upper2 = np.array([160, 30, 20]), np.array([180, 255, 255])
    return cv2.inRange(hsv, lower1, upper1) + cv2.inRange(hsv, lower2, upper2)

def detectar_descontinuidade_borda(mascara_objeto, img_resultado, area_min_quebra_local):
    contornos, _ = cv2.findContours(mascara_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos: return False, np.zeros_like(mascara_objeto)

    maior_contorno = max(contornos, key=cv2.contourArea)
    epsilon = 0.005 * cv2.arcLength(maior_contorno, True)
    contorno_suavizado = cv2.approxPolyDP(maior_contorno, epsilon, True)
    hull = cv2.convexHull(contorno_suavizado, returnPoints=False)
    
    descontinuidades = 0
    img_alertas = np.zeros_like(mascara_objeto)

    if hull is not None and len(hull) > 3 and len(contorno_suavizado) > 3:
        defeitos = cv2.convexityDefects(contorno_suavizado, hull)
        if defeitos is not None:
            for i in range(defeitos.shape[0]):
                s, e, f, d = defeitos[i, 0]
                profundidade = d / 256.0
                start = tuple(contorno_suavizado[s][0])
                end = tuple(contorno_suavizado[e][0])
                ponto_fundo = tuple(contorno_suavizado[f][0])
                largura_defeito = np.sqrt((start[0] - end[0])**2 + (start[1] - end[1])**2)
                
                if profundidade > 15 and largura_defeito < 150:
                    area_aproximada = (largura_defeito * profundidade) / 2
                    if area_aproximada > area_min_quebra_local:
                        descontinuidades += 1
                        cv2.circle(img_resultado, ponto_fundo, 15, (0, 0, 255), 3)
                        cv2.putText(img_resultado, "QUEBRA", (ponto_fundo[0] + 20, ponto_fundo[1]), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                        cv2.circle(img_alertas, ponto_fundo, 15, 255, -1)

    return (descontinuidades == 0), img_alertas
    
def analisar_roi_robusto(imagem_roi, sensibilidade_ajuste, tol_pixels, area_min_quebra_local):
    img_resultado = imagem_roi.copy()
    mascara_cor = segmentar_vermelho(imagem_roi)
    kernel_limpeza = np.ones((7, 7), np.uint8)
    mascara_objeto = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel_limpeza)
    mascara_objeto = cv2.morphologyEx(mascara_objeto, cv2.MORPH_CLOSE, kernel_limpeza)
    mascara_objeto = cv2.dilate(mascara_objeto, np.ones((5, 5), np.uint8), iterations=1) 
    
    aprovado_borda, img_alertas_borda = detectar_descontinuidade_borda(mascara_objeto, img_resultado, area_min_quebra_local)
    
    cinza = cv2.cvtColor(imagem_roi, cv2.COLOR_BGR2GRAY)
    kernel_blackhat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(cinza, cv2.MORPH_BLACKHAT, kernel_blackhat)
    
    limiar_base = max(5, min(25 - sensibilidade_ajuste, 255))
    _, trincas_binarias = cv2.threshold(blackhat, limiar_base, 255, cv2.THRESH_BINARY)
    
    trincas_no_objeto = cv2.bitwise_and(trincas_binarias, mascara_objeto)
    contornos_trincas, _ = cv2.findContours(trincas_no_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara_trincas_finais = np.zeros_like(trincas_no_objeto)
    
    for cnt in contornos_trincas:
        area_contorno = cv2.contourArea(cnt)
        if area_contorno > 10: 
            rect = cv2.minAreaRect(cnt)
            largura, altura = rect[1]
            if largura > 0 and altura > 0:
                proporcao = max(largura, altura) / min(largura, altura)
                hull = cv2.convexHull(cnt)
                hull_area = cv2.contourArea(hull)
                solidity = area_contorno / hull_area if hull_area > 0 else 0
                
                if proporcao > 1.5 and solidity < 0.9:
                    cv2.drawContours(mascara_trincas_finais, [cnt], -1, 255, thickness=cv2.FILLED)

    area_falha_px = cv2.countNonZero(mascara_trincas_finais)
    aprovado_superficie = (area_falha_px <= tol_pixels)
    aprovado_geral = aprovado_superficie and aprovado_borda

    img_resultado[mascara_trincas_finais == 255] = [0, 255, 255] 
    
    imagens_debug = {
        "1. Mascara Cor": mascara_cor,
        "2. Falhas Borda": img_alertas_borda,
        "3. Trincas": blackhat,
        "4. Filtro Final": mascara_trincas_finais
    }
    
    return img_resultado, aprovado_geral, imagens_debug

def mostrar_painel_debug(imagens_debug):
    nomes = list(imagens_debug.keys())
    imgs_bgr = [cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img for img in imagens_debug.values()]
    for i in range(4): cv2.putText(imgs_bgr[i], nomes[i], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    painel = np.vstack((np.hstack((imgs_bgr[0], imgs_bgr[1])), np.hstack((imgs_bgr[2], imgs_bgr[3]))))
    cv2.imshow("Painel de Debug CV", cv2.resize(painel, (600, 600)))

def main():
    global sensibilidade, tolerancia_px, area_min_quebra
    
    cap = cv2.VideoCapture(0)
    analisar = False
    frame_analisado = None
    ultimo_debug = None
    
    # Cria a janela do OpenCV e vincula a função de clique do mouse
    cv2.namedWindow("Inspecao de Qualidade")
    cv2.setMouseCallback("Inspecao de Qualidade", controle_mouse)
    
    largura_roi = 300
    altura_roi = 300
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        altura_tela, largura_tela = frame.shape[:2]
        x1, y1 = (largura_tela - largura_roi) // 2, (altura_tela - altura_roi) // 2
        x2, y2 = x1 + largura_roi, y1 + altura_roi
            
        if analisar and frame_analisado is not None:
            # Desenha a interface por cima da imagem processada
            frame_final = desenhar_painel_sobreposto(frame_analisado)
            cv2.imshow("Inspecao de Qualidade", frame_final)
            if ultimo_debug: mostrar_painel_debug(ultimo_debug)
        else:
            frame_exibicao = frame.copy()
            cv2.rectangle(frame_exibicao, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame_exibicao, "Posicione a peca e aperte ESPACO", (x1 - 30, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            
            # Desenha a interface interativa por cima da câmera ao vivo
            frame_exibicao = desenhar_painel_sobreposto(frame_exibicao)
            cv2.imshow("Inspecao de Qualidade", frame_exibicao)
            
        tecla = cv2.waitKey(1) & 0xFF
        
        if tecla == 27: break
        elif tecla == 32: 
            if analisar: 
                analisar = False 
                try: cv2.destroyWindow("Painel de Debug CV") 
                except: pass
            else:
                roi = frame[y1:y2, x1:x2]
                
                # Passa as variáveis globais que o mouse altera
                roi_processado, aprovado, ultimo_debug = analisar_roi_robusto(
                    roi, sensibilidade, tolerancia_px, area_min_quebra
                )
                
                frame_analisado = frame.copy()
                frame_analisado[y1:y2, x1:x2] = roi_processado
                
                cor = (0, 255, 0) if aprovado else (0, 0, 255)
                texto = "APROVADO" if aprovado else "REPROVADO"
                cv2.rectangle(frame_analisado, (x1, y1), (x2, y2), cor, 3)
                cv2.putText(frame_analisado, texto, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 3)
                
                if not aprovado:
                    data_hora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                    nome_arquivo = f"falhas/reprovado_{data_hora}.jpg"
                    cv2.imwrite(nome_arquivo, frame_analisado)
                
                analisar = True
                
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Versão 0.2.0

In [16]:
import cv2
import numpy as np
import os
import datetime

# --- Variáveis Globais de Calibração ---
sensibilidade = 0
tolerancia_px = 40

# Cria o diretório para salvar as falhas
if not os.path.exists('falhas'):
    os.makedirs('falhas')

def controle_mouse(event, x, y, flags, param):
    """
    Detecta cliques do mouse na tela para ajustar os valores.
    Coordenadas (x, y) precisam bater com os retângulos desenhados na interface.
    """
    global sensibilidade, tolerancia_px
    
    if event == cv2.EVENT_LBUTTONDOWN:
        # Linha 1: Sensibilidade (Y entre 20 e 50)
        if 20 <= y <= 50:
            if 300 <= x <= 340: sensibilidade = max(0, sensibilidade - 1)
            elif 360 <= x <= 400: sensibilidade = min(20, sensibilidade + 1)
            
        # Linha 2: Tolerância Superfície (Y entre 60 e 90)
        elif 60 <= y <= 90:
            if 300 <= x <= 340: tolerancia_px = max(0, tolerancia_px - 5)
            elif 360 <= x <= 400: tolerancia_px = min(300, tolerancia_px + 5)

def desenhar_painel_sobreposto(frame):
    """
    Desenha o painel de controle escuro com botões sobre a imagem da câmera.
    """
    cv2.rectangle(frame, (10, 10), (420, 105), (30, 30, 30), -1)
    cv2.rectangle(frame, (10, 10), (420, 105), (200, 200, 200), 1) 

    linhas = [
        (f"Sensib. BlackHat: {sensibilidade}", 40, 20),
        (f"Tol. Superficie (px): {tolerancia_px}", 80, 60)
    ]

    for texto, y_texto, y_botao in linhas:
        cv2.putText(frame, texto, (20, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.rectangle(frame, (300, y_botao), (340, y_botao + 30), (0, 0, 180), -1)
        cv2.putText(frame, "-", (312, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.rectangle(frame, (360, y_botao), (400, y_botao + 30), (0, 180, 0), -1)
        cv2.putText(frame, "+", (372, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
    return frame

def segmentar_vermelho(img_bgr):
    img_suavizada = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv = cv2.cvtColor(img_suavizada, cv2.COLOR_BGR2HSV)
    lower1, upper1 = np.array([0, 30, 20]), np.array([10, 255, 255])
    lower2, upper2 = np.array([160, 30, 20]), np.array([180, 255, 255])
    return cv2.inRange(hsv, lower1, upper1) + cv2.inRange(hsv, lower2, upper2)

def analisar_roi_robusto(imagem_roi, sensibilidade_ajuste, tol_pixels):
    img_resultado = imagem_roi.copy()
    mascara_cor = segmentar_vermelho(imagem_roi)
    kernel_limpeza = np.ones((7, 7), np.uint8)
    
    mascara_objeto = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel_limpeza)
    mascara_objeto = cv2.morphologyEx(mascara_objeto, cv2.MORPH_CLOSE, kernel_limpeza)
    mascara_objeto = cv2.dilate(mascara_objeto, np.ones((5, 5), np.uint8), iterations=1) 
    
    cinza = cv2.cvtColor(imagem_roi, cv2.COLOR_BGR2GRAY)
    kernel_blackhat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(cinza, cv2.MORPH_BLACKHAT, kernel_blackhat)
    
    limiar_base = max(5, min(25 - sensibilidade_ajuste, 255))
    _, trincas_binarias = cv2.threshold(blackhat, limiar_base, 255, cv2.THRESH_BINARY)
    
    trincas_no_objeto = cv2.bitwise_and(trincas_binarias, mascara_objeto)
    contornos_trincas, _ = cv2.findContours(trincas_no_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara_trincas_finais = np.zeros_like(trincas_no_objeto)
    
    for cnt in contornos_trincas:
        area_contorno = cv2.contourArea(cnt)
        if area_contorno > 10: 
            rect = cv2.minAreaRect(cnt)
            largura, altura = rect[1]
            if largura > 0 and altura > 0:
                proporcao = max(largura, altura) / min(largura, altura)
                hull = cv2.convexHull(cnt)
                hull_area = cv2.contourArea(hull)
                solidity = area_contorno / hull_area if hull_area > 0 else 0
                
                if proporcao > 1.5 and solidity < 0.9:
                    cv2.drawContours(mascara_trincas_finais, [cnt], -1, 255, thickness=cv2.FILLED)

    area_falha_px = cv2.countNonZero(mascara_trincas_finais)
    aprovado_geral = (area_falha_px <= tol_pixels)

    img_resultado[mascara_trincas_finais == 255] = [0, 255, 255] 
    
    imagens_debug = {
        "1. Mascara Cor": mascara_cor,
        "2. Mascara Objeto": mascara_objeto,
        "3. Filtro BlackHat": blackhat,
        "4. Trincas Finais": mascara_trincas_finais
    }
    
    return img_resultado, aprovado_geral, imagens_debug

def mostrar_painel_debug(imagens_debug):
    nomes = list(imagens_debug.keys())
    imgs_bgr = [cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img for img in imagens_debug.values()]
    for i in range(4): cv2.putText(imgs_bgr[i], nomes[i], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    painel = np.vstack((np.hstack((imgs_bgr[0], imgs_bgr[1])), np.hstack((imgs_bgr[2], imgs_bgr[3]))))
    cv2.imshow("Painel de Debug CV", cv2.resize(painel, (600, 600)))

def main():
    global sensibilidade, tolerancia_px
    
    cap = cv2.VideoCapture(0)
    falha_ja_salva = False # Impede que o sistema salve 30 fotos por segundo da mesma falha
    
    cv2.namedWindow("Inspecao de Qualidade")
    cv2.setMouseCallback("Inspecao de Qualidade", controle_mouse)
    
    largura_roi = 300
    altura_roi = 300
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        altura_tela, largura_tela = frame.shape[:2]
        x1, y1 = (largura_tela - largura_roi) // 2, (altura_tela - altura_roi) // 2
        x2, y2 = x1 + largura_roi, y1 + altura_roi
        
        # Extrai o ROI (agora analisamos ele diretamente, sem checar se tem peça)
        roi_vivo = frame[y1:y2, x1:x2]
        frame_exibicao = frame.copy()
        
        # Analisa o ROI continuamente
        roi_processado, aprovado, ultimo_debug = analisar_roi_robusto(
            roi_vivo, sensibilidade, tolerancia_px
        )
        
        # Substitui a área central pela imagem processada com as marcações
        frame_exibicao[y1:y2, x1:x2] = roi_processado
        
        cor = (0, 255, 0) if aprovado else (0, 0, 255)
        texto = "APROVADO" if aprovado else "REPROVADO"
        
        cv2.rectangle(frame_exibicao, (x1, y1), (x2, y2), cor, 3)
        cv2.putText(frame_exibicao, texto, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 4)
        
        # Atualiza o painel de debug ao vivo
        mostrar_painel_debug(ultimo_debug)
        
        # Lógica para salvar imagens de falha de forma controlada
        if not aprovado:
            if not falha_ja_salva:
                data_hora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                nome_arquivo = f"falhas/reprovado_{data_hora}.jpg"
                cv2.imwrite(nome_arquivo, frame_exibicao)
                falha_ja_salva = True # Trava para não salvar o próximo frame do mesmo defeito
        else:
            # Se for aprovado (peça boa ou fundo vazio sem ruídos), rearma a trava de salvamento
            falha_ja_salva = False
            
        # Desenha o painel de controles por cima de tudo
        frame_final = desenhar_painel_sobreposto(frame_exibicao)
        cv2.imshow("Inspecao de Qualidade", frame_final)
            
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27: # Tecla ESC para sair
            break
                
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

In [12]:
import cv2
import numpy as np
import os
import datetime

# --- Variáveis Globais de Calibração ---
sensibilidade = 0
tolerancia_px = 40

# Cria o diretório para salvar as falhas
if not os.path.exists('falhas'):
    os.makedirs('falhas')

def controle_mouse(event, x, y, flags, param):
    """
    Detecta cliques do mouse na tela para ajustar os valores.
    Coordenadas (x, y) precisam bater com os retângulos desenhados na interface.
    """
    global sensibilidade, tolerancia_px
    
    if event == cv2.EVENT_LBUTTONDOWN:
        # Linha 1: Sensibilidade (Y entre 20 e 50)
        if 20 <= y <= 50:
            if 300 <= x <= 340: sensibilidade = max(0, sensibilidade - 1)
            elif 360 <= x <= 400: sensibilidade = min(20, sensibilidade + 1)
            
        # Linha 2: Tolerância Superfície (Y entre 60 e 90)
        elif 60 <= y <= 90:
            if 300 <= x <= 340: tolerancia_px = max(0, tolerancia_px - 5)
            elif 360 <= x <= 400: tolerancia_px = min(300, tolerancia_px + 5)

def desenhar_painel_sobreposto(frame):
    """
    Desenha o painel de controle escuro com botões sobre a imagem da câmera.
    """
    cv2.rectangle(frame, (10, 10), (420, 105), (30, 30, 30), -1)
    cv2.rectangle(frame, (10, 10), (420, 105), (200, 200, 200), 1) 

    linhas = [
        (f"Sensib. BlackHat: {sensibilidade}", 40, 20),
        (f"Tol. Superficie (px): {tolerancia_px}", 80, 60)
    ]

    for texto, y_texto, y_botao in linhas:
        cv2.putText(frame, texto, (20, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.rectangle(frame, (300, y_botao), (340, y_botao + 30), (0, 0, 180), -1)
        cv2.putText(frame, "-", (312, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.rectangle(frame, (360, y_botao), (400, y_botao + 30), (0, 180, 0), -1)
        cv2.putText(frame, "+", (372, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
    return frame

def segmentar_cor_predominante(img_bgr):
    """
    Encontra a cor com maior incidência no ROI e gera uma máscara dinâmica.
    Ignora fundo preto, branco e cinza.
    """
    img_suavizada = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv = cv2.cvtColor(img_suavizada, cv2.COLOR_BGR2HSV)
    
    # Cria uma máscara isolando apenas pixels que têm alguma cor real
    # Sat > 40 e Val > 40 ignoram preto profundo, branco estourado e cinzas
    mask_validos = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([180, 255, 255]))
    
    # Calcula o histograma do canal Matiz (Hue) usando apenas os pixels válidos
    hist = cv2.calcHist([hsv], [0], mask_validos, [180], [0, 180])
    
    # Encontra qual valor de Matiz tem o pico (cor predominante)
    _, max_val, _, max_loc = cv2.minMaxLoc(hist)
    
    # Se a imagem for inteira preta/branca/cinza (nenhum pixel válido)
    if max_val == 0:
        return np.zeros(hsv.shape[:2], dtype=np.uint8)
        
    dominant_hue = max_loc[1]
    hue_tol = 15 # Tolerância do matiz (quanto maior, mais tons da mesma cor ele pega)
    
    # Calcula os limites e resolve a quebra de ciclo no OpenCV (Hue vai de 0 a 180)
    lower_hue = dominant_hue - hue_tol
    upper_hue = dominant_hue + hue_tol
    
    if lower_hue < 0:
        mask1 = cv2.inRange(hsv, np.array([180 + lower_hue, 40, 40]), np.array([180, 255, 255]))
        mask2 = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([upper_hue, 255, 255]))
        return cv2.bitwise_or(mask1, mask2)
    elif upper_hue > 180:
        mask1 = cv2.inRange(hsv, np.array([lower_hue, 40, 40]), np.array([180, 255, 255]))
        mask2 = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([upper_hue - 180, 255, 255]))
        return cv2.bitwise_or(mask1, mask2)
    else:
        return cv2.inRange(hsv, np.array([lower_hue, 40, 40]), np.array([upper_hue, 255, 255]))

def analisar_roi_robusto(imagem_roi, sensibilidade_ajuste, tol_pixels):
    img_resultado = imagem_roi.copy()
    
    # Mudança para função que detecta qualquer cor
    mascara_cor = segmentar_cor_predominante(imagem_roi)
    
    kernel_limpeza = np.ones((7, 7), np.uint8)
    
    mascara_objeto = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel_limpeza)
    mascara_objeto = cv2.morphologyEx(mascara_objeto, cv2.MORPH_CLOSE, kernel_limpeza)
    mascara_objeto = cv2.dilate(mascara_objeto, np.ones((5, 5), np.uint8), iterations=1) 
    
    cinza = cv2.cvtColor(imagem_roi, cv2.COLOR_BGR2GRAY)
    kernel_blackhat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(cinza, cv2.MORPH_BLACKHAT, kernel_blackhat)
    
    limiar_base = max(5, min(25 - sensibilidade_ajuste, 255))
    _, trincas_binarias = cv2.threshold(blackhat, limiar_base, 255, cv2.THRESH_BINARY)
    
    trincas_no_objeto = cv2.bitwise_and(trincas_binarias, mascara_objeto)
    contornos_trincas, _ = cv2.findContours(trincas_no_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara_trincas_finais = np.zeros_like(trincas_no_objeto)
    
    for cnt in contornos_trincas:
        area_contorno = cv2.contourArea(cnt)
        if area_contorno > 10: 
            rect = cv2.minAreaRect(cnt)
            largura, altura = rect[1]
            if largura > 0 and altura > 0:
                proporcao = max(largura, altura) / min(largura, altura)
                hull = cv2.convexHull(cnt)
                hull_area = cv2.contourArea(hull)
                solidity = area_contorno / hull_area if hull_area > 0 else 0
                
                if proporcao > 1.5 and solidity < 0.9:
                    cv2.drawContours(mascara_trincas_finais, [cnt], -1, 255, thickness=cv2.FILLED)

    area_falha_px = cv2.countNonZero(mascara_trincas_finais)
    aprovado_geral = (area_falha_px <= tol_pixels)

    img_resultado[mascara_trincas_finais == 255] = [0, 255, 255] 
    
    imagens_debug = {
        "1. Mascara Cor (Auto)": mascara_cor,
        "2. Mascara Objeto": mascara_objeto,
        "3. Filtro BlackHat": blackhat,
        "4. Trincas Finais": mascara_trincas_finais
    }
    
    return img_resultado, aprovado_geral, imagens_debug

def mostrar_painel_debug(imagens_debug):
    nomes = list(imagens_debug.keys())
    imgs_bgr = [cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img for img in imagens_debug.values()]
    for i in range(4): cv2.putText(imgs_bgr[i], nomes[i], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    painel = np.vstack((np.hstack((imgs_bgr[0], imgs_bgr[1])), np.hstack((imgs_bgr[2], imgs_bgr[3]))))
    cv2.imshow("Painel de Debug CV", cv2.resize(painel, (600, 600)))

def main():
    global sensibilidade, tolerancia_px
    
    cap = cv2.VideoCapture(0)
    falha_ja_salva = False 
    
    cv2.namedWindow("Inspecao de Qualidade")
    cv2.setMouseCallback("Inspecao de Qualidade", controle_mouse)
    
    largura_roi = 300
    altura_roi = 300
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        altura_tela, largura_tela = frame.shape[:2]
        x1, y1 = (largura_tela - largura_roi) // 2, (altura_tela - altura_roi) // 2
        x2, y2 = x1 + largura_roi, y1 + altura_roi
        
        roi_vivo = frame[y1:y2, x1:x2]
        frame_exibicao = frame.copy()
        
        roi_processado, aprovado, ultimo_debug = analisar_roi_robusto(
            roi_vivo, sensibilidade, tolerancia_px
        )
        
        frame_exibicao[y1:y2, x1:x2] = roi_processado
        
        cor = (0, 255, 0) if aprovado else (0, 0, 255)
        texto = "APROVADO" if aprovado else "REPROVADO"
        
        cv2.rectangle(frame_exibicao, (x1, y1), (x2, y2), cor, 3)
        cv2.putText(frame_exibicao, texto, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 4)
        
        mostrar_painel_debug(ultimo_debug)
        
        if not aprovado:
            if not falha_ja_salva:
                data_hora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                nome_arquivo = f"falhas/reprovado_{data_hora}.jpg"
                cv2.imwrite(nome_arquivo, frame_exibicao)
                falha_ja_salva = True 
        else:
            falha_ja_salva = False
            
        frame_final = desenhar_painel_sobreposto(frame_exibicao)
        cv2.imshow("Inspecao de Qualidade", frame_final)
            
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27: 
            break
                
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

In [2]:
import cv2
import numpy as np
import os
import datetime

# --- Variáveis Globais de Calibração ---
sensibilidade = 0
tolerancia_px = 40

# Cria o diretório para salvar as falhas
if not os.path.exists('falhas'):
    os.makedirs('falhas')

def controle_mouse(event, x, y, flags, param):
    """
    Detecta cliques do mouse na tela para ajustar os valores.
    Coordenadas (x, y) precisam bater com os retângulos desenhados na interface.
    """
    global sensibilidade, tolerancia_px
    
    if event == cv2.EVENT_LBUTTONDOWN:
        # Linha 1: Sensibilidade (Y entre 20 e 50)
        if 20 <= y <= 50:
            if 300 <= x <= 340: sensibilidade = max(0, sensibilidade - 1)
            elif 360 <= x <= 400: sensibilidade = min(20, sensibilidade + 1)
            
        # Linha 2: Tolerância Superfície (Y entre 60 e 90)
        elif 60 <= y <= 90:
            if 300 <= x <= 340: tolerancia_px = max(0, tolerancia_px - 5)
            elif 360 <= x <= 400: tolerancia_px = min(300, tolerancia_px + 5)

def desenhar_painel_sobreposto(frame):
    """
    Desenha o painel de controle escuro com botões sobre a imagem da câmera.
    """
    cv2.rectangle(frame, (10, 10), (420, 105), (30, 30, 30), -1)
    cv2.rectangle(frame, (10, 10), (420, 105), (200, 200, 200), 1) 

    linhas = [
        (f"Sensib. BlackHat: {sensibilidade}", 40, 20),
        (f"Tol. Superficie (px): {tolerancia_px}", 80, 60)
    ]

    for texto, y_texto, y_botao in linhas:
        cv2.putText(frame, texto, (20, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.rectangle(frame, (300, y_botao), (340, y_botao + 30), (0, 0, 180), -1)
        cv2.putText(frame, "-", (312, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.rectangle(frame, (360, y_botao), (400, y_botao + 30), (0, 180, 0), -1)
        cv2.putText(frame, "+", (372, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
    return frame

def segmentar_cor_central(img_bgr):
    """
    Pega a cor exata do pixel central do ROI e cria uma máscara baseada nela.
    """
    # Suaviza para evitar que um único pixel de ruído bagunce a leitura
    img_suavizada = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv = cv2.cvtColor(img_suavizada, cv2.COLOR_BGR2HSV)
    
    altura, largura = hsv.shape[:2]
    cy, cx = altura // 2, largura // 2
    
    # Pega o valor HSV do pixel central
    pixel_central_hsv = hsv[cy, cx]
    
    # Se o pixel central for muito escuro ou muito desbotado, ignora (fundo vazio)
    if pixel_central_hsv[1] < 40 or pixel_central_hsv[2] < 40:
         return np.zeros(hsv.shape[:2], dtype=np.uint8)
         
    cor_alvo_hue = int(pixel_central_hsv[0])
    hue_tol = 15 # Tolerância do matiz
    
    # Calcula os limites com proteção para a virada do ciclo de cores (180 graus)
    lower_hue = cor_alvo_hue - hue_tol
    upper_hue = cor_alvo_hue + hue_tol
    
    if lower_hue < 0:
        mask1 = cv2.inRange(hsv, np.array([180 + lower_hue, 40, 40]), np.array([180, 255, 255]))
        mask2 = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([upper_hue, 255, 255]))
        mask = cv2.bitwise_or(mask1, mask2)
    elif upper_hue > 180:
        mask1 = cv2.inRange(hsv, np.array([lower_hue, 40, 40]), np.array([180, 255, 255]))
        mask2 = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([upper_hue - 180, 255, 255]))
        mask = cv2.bitwise_or(mask1, mask2)
    else:
        mask = cv2.inRange(hsv, np.array([lower_hue, 40, 40]), np.array([upper_hue, 255, 255]))
        
    # Opcional: desenha um pontinho no centro da imagem_bgr para você ver onde ele está lendo a cor
    cv2.circle(img_bgr, (cx, cy), 3, (0, 255, 0), -1)
    
    return mask

def analisar_roi_robusto(imagem_roi, sensibilidade_ajuste, tol_pixels):
    img_resultado = imagem_roi.copy()
    
    # Segmentação base
    mascara_cor = segmentar_cor_central(imagem_roi)
    
    kernel_limpeza = np.ones((7, 7), np.uint8)
    mascara_objeto = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel_limpeza)
    mascara_objeto = cv2.morphologyEx(mascara_objeto, cv2.MORPH_CLOSE, kernel_limpeza)
    
    # 1. REMOVIDO: cv2.dilate (que estava empurrando a máscara para a sombra do fundo)
    
    # 2. ADICIONADO: Erosão para criar uma margem de segurança nas bordas
    # Quanto maior o kernel (ex: 15, 15), mais distante da borda ele ignora as falhas.
    kernel_margem = np.ones((15, 15), np.uint8)
    mascara_analise = cv2.erode(mascara_objeto, kernel_margem, iterations=1)
    
    # Filtro BlackHat
    cinza = cv2.cvtColor(imagem_roi, cv2.COLOR_BGR2GRAY)
    kernel_blackhat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(cinza, cv2.MORPH_BLACKHAT, kernel_blackhat)
    
    limiar_base = max(5, min(25 - sensibilidade_ajuste, 255))
    _, trincas_binarias = cv2.threshold(blackhat, limiar_base, 255, cv2.THRESH_BINARY)
    
    # 3. ALTERADO: Cruza o blackhat com a máscara ENCOLHIDA (mascara_analise)
    trincas_no_objeto = cv2.bitwise_and(trincas_binarias, mascara_analise)
    contornos_trincas, _ = cv2.findContours(trincas_no_objeto, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mascara_trincas_finais = np.zeros_like(trincas_no_objeto)
    
    for cnt in contornos_trincas:
        area_contorno = cv2.contourArea(cnt)
        if area_contorno > 10: 
            rect = cv2.minAreaRect(cnt)
            largura, altura = rect[1]
            if largura > 0 and altura > 0:
                proporcao = max(largura, altura) / min(largura, altura)
                hull = cv2.convexHull(cnt)
                hull_area = cv2.contourArea(hull)
                solidity = area_contorno / hull_area if hull_area > 0 else 0
                
                if proporcao > 1.5 and solidity < 0.9:
                    cv2.drawContours(mascara_trincas_finais, [cnt], -1, 255, thickness=cv2.FILLED)

    area_falha_px = cv2.countNonZero(mascara_trincas_finais)
    aprovado_geral = (area_falha_px <= tol_pixels)

    img_resultado[mascara_trincas_finais == 255] = [0, 255, 255] 
    
    imagens_debug = {
        "1. Mascara Cor (Centro)": mascara_cor,
        "2. Mascara Analise (Com Margem)": mascara_analise, # Atualizado no debug para você ver a margem
        "3. Filtro BlackHat": blackhat,
        "4. Trincas Finais": mascara_trincas_finais
    }
    
    return img_resultado, aprovado_geral, imagens_debug

def mostrar_painel_debug(imagens_debug):
    nomes = list(imagens_debug.keys())
    imgs_bgr = [cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img for img in imagens_debug.values()]
    for i in range(4): cv2.putText(imgs_bgr[i], nomes[i], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    painel = np.vstack((np.hstack((imgs_bgr[0], imgs_bgr[1])), np.hstack((imgs_bgr[2], imgs_bgr[3]))))
    cv2.imshow("Painel de Debug CV", cv2.resize(painel, (600, 600)))

def main():
    global sensibilidade, tolerancia_px
    
    cap = cv2.VideoCapture(0)
    falha_ja_salva = False 
    
    cv2.namedWindow("Inspecao de Qualidade")
    cv2.setMouseCallback("Inspecao de Qualidade", controle_mouse)
    
    largura_roi = 300
    altura_roi = 300
    
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        altura_tela, largura_tela = frame.shape[:2]
        x1, y1 = (largura_tela - largura_roi) // 2, (altura_tela - altura_roi) // 2
        x2, y2 = x1 + largura_roi, y1 + altura_roi
        
        roi_vivo = frame[y1:y2, x1:x2]
        frame_exibicao = frame.copy()
        
        roi_processado, aprovado, ultimo_debug = analisar_roi_robusto(
            roi_vivo, sensibilidade, tolerancia_px
        )
        
        # Desenha uma pequena mira em cruz no centro do ROI na tela principal para guiar o usuário
        centro_x_tela = x1 + (largura_roi // 2)
        centro_y_tela = y1 + (altura_roi // 2)
        cv2.drawMarker(frame_exibicao, (centro_x_tela, centro_y_tela), (0, 255, 0), markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
        
        frame_exibicao[y1:y2, x1:x2] = roi_processado
        
        cor = (0, 255, 0) if aprovado else (0, 0, 255)
        texto = "APROVADO" if aprovado else "REPROVADO"
        
        cv2.rectangle(frame_exibicao, (x1, y1), (x2, y2), cor, 3)
        cv2.putText(frame_exibicao, texto, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 4)
        
        mostrar_painel_debug(ultimo_debug)
        
        if not aprovado:
            if not falha_ja_salva:
                data_hora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                nome_arquivo = f"falhas/reprovado_{data_hora}.jpg"
                cv2.imwrite(nome_arquivo, frame_exibicao)
                falha_ja_salva = True 
        else:
            falha_ja_salva = False
            
        frame_final = desenhar_painel_sobreposto(frame_exibicao)
        cv2.imshow("Inspecao de Qualidade", frame_final)
            
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27: 
            break
                
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()